In [1]:
import os
import json
import subprocess
import shutil

# Save the initial current directory
initial_dir = os.getcwd()
print(f"Starting script in directory: {initial_dir}")

# Define the path to the hosts file
HOSTS_FILE = './cloudflare/hosts.json'

# Check if the hosts file exists
if not os.path.isfile(HOSTS_FILE):
    print(f"Hosts file not found: {HOSTS_FILE}")
    exit(1)

# Load hosts from JSON file
with open(HOSTS_FILE) as f:
    hosts = json.load(f)

# Define GitHub user and branch
GITHUB_USER = "allwomenstalk"
BRANCH = "main"  # Change this to the branch you want to deploy to




Starting script in directory: /Users/slavik/Documents/code/allwomenstalk/11ty/njkarchives


In [2]:
# Function to run a shell command
def run_command(command):
    print(f"Running command: {command}")
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return result.stdout.decode('utf-8').strip()
    except subprocess.CalledProcessError as e:
        print(f"Command '{command}' failed with exit status {e.returncode}")
        print(f"stdout: {e.stdout.decode('utf-8').strip()}")
        print(f"stderr: {e.stderr.decode('utf-8').strip()}")
        raise

In [4]:
# Loop through each host
for host in hosts:
    DOMAIN = host['domain']
    BUILD_DIR = f"_site/{DOMAIN}"
    REPO_NAME = DOMAIN
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

    print(f"Deploying site to GitHub Pages for {DOMAIN}...")

    # Check if the build directory exists
    if not os.path.isdir(BUILD_DIR):
        print(f"Build directory {BUILD_DIR} does not exist. Skipping...")
        continue

    # Navigate to the build directory
    os.chdir(BUILD_DIR)

    # Print the current directory after changing it
    print(f"Current directory after changing: {os.getcwd()}")

    # Initialize a new git repository if it doesn't exist
    if not os.path.isdir(".git"):
        run_command("git init")

    # Remove any existing remote named origin and add the new one
    try:
        run_command("git remote remove origin")
    except subprocess.CalledProcessError as e:
        if "No such remote" in e.stderr.decode('utf-8'):
            print("Remote 'origin' does not exist, skipping removal.")
        else:
            raise

    run_command(f"git remote add origin {REPO_URL}")

    # Add all files to the new repository
    run_command("git add .")

    # Commit the changes
    try:
        run_command("git diff-index --quiet HEAD")
        print("Nothing to commit, working tree clean.")
    except subprocess.CalledProcessError:
        run_command('git commit -m "Deploy site to GitHub Pages"')

    # Create and switch to the specified branch if it doesn't exist
    run_command(f"git checkout -B {BRANCH}")

    # Push the changes to the specified branch, creating it if it doesn't exist
    run_command(f"git push --set-upstream origin {BRANCH} --force")

    # Return to the original directory
    os.chdir(initial_dir)

    # Cleanup: Remove the .git directory to reset the state
    shutil.rmtree(os.path.join(BUILD_DIR, ".git"))

    print(f"Deployment complete for {DOMAIN}.")
    print(f"Resetting to initial directory: {initial_dir}")

Deploying site to GitHub Pages for hair.allwomenstalk.com...
Current directory after changing: /Users/slavik/Documents/code/allwomenstalk/11ty/njkarchives/_site/hair.allwomenstalk.com
Running command: git init
Running command: git remote remove origin
Command 'git remote remove origin' failed with exit status 2
stdout: 
stderr: error: No such remote: 'origin'
Remote 'origin' does not exist, skipping removal.
Running command: git remote add origin https://github.com/allwomenstalk/hair.allwomenstalk.com.git
Running command: git add .
Running command: git diff-index --quiet HEAD
Command 'git diff-index --quiet HEAD' failed with exit status 128
stdout: 
stderr: fatal: ambiguous argument 'HEAD': unknown revision or path not in the working tree.
Use '--' to separate paths from revisions, like this:
'git <command> [<revision>...] -- [<file>...]'
Running command: git commit -m "Deploy site to GitHub Pages"
Running command: git checkout -B main
Running command: git push --set-upstream origin ma